<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Algopro/NLP%20Embedding%20student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
VOCAB_SIZE = 20000
EMBEDDING_DIM = 100

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load QQP
qqp = load_dataset("glue", "qqp")
train_df = qqp["train"].to_pandas().dropna(subset=["question1", "question2"]).reset_index(drop=True)
val_df = qqp["validation"].to_pandas().dropna(subset=["question1", "question2"]).reset_index(drop=True)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
def preprocess(text):
    ...

In [ ]:
# create vocabulary

word2idx =

In [ ]:
def text_to_indices(text, word2idx):
    ...

In [ ]:
class QQPDataset(Dataset):
    def __init__(self, df, word2idx):
        self.labels = df["label"].tolist()
        self.q1 = [text_to_indices(t, word2idx) for t in df["question1"]]
        self.q2 = [text_to_indices(t, word2idx) for t in df["question2"]]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.q1[idx], dtype=torch.long),
            torch.tensor(self.q2[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float),
        )

In [ ]:
def collate_fn(batch):
    q1s, q2s, labels = zip(*batch)
    q1s = nn.utils.rnn.pad_sequence(q1s, batch_first=True, padding_value=0)
    q2s = nn.utils.rnn.pad_sequence(q2s, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return q1s, q2s, labels

train_loader = DataLoader(QQPDataset(train_df, word2idx), batch_size=256, collate_fn=collate_fn, shuffle=True)
val_loader = DataLoader(QQPDataset(val_df, word2idx), batch_size=256, collate_fn=collate_fn)

In [ ]:
class QQPModel(nn.Module):
    def __init__(self, num_words, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_words, embed_dim, padding_idx=0)
        # Input: concat of q1, q2, q1*q2 = 3 * embed_dim
        self.classifier =

    def encode(self, x):
        # encode a text into a single embedding (average the embedding of all non padding words)
        ...

    def forward(self, q1, q2):
        ...

In [ ]:
model = QQPModel(VOCAB_SIZE, EMBEDDING_DIM).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), 0.0001)

for epoch in range(10):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for q1, q2, labels in train_loader:
        ...

    model.eval()
    v_correct = v_total = 0
    with torch.no_grad():
        for q1, q2, labels in val_loader:
            q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
            out = model(q1, q2)
            v_correct += ((out > 0).float() == labels).sum().item()
            v_total += len(labels)

    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/total:.4f} | Train: {correct/total:.4f} | Val: {v_correct/v_total:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for q1, q2, labels in val_loader:
        q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
        out = model(q1, q2)
        preds = (out > 0).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=["Not Duplicate", "Duplicate"],
    digits=4
))